In [1]:
import pandas as pd
from bokeh.io import output_notebook, show, push_notebook
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, HoverTool
from ipywidgets import interact, IntSlider

output_notebook()

Loading BokehJS ...

In [2]:
df = pd.read_csv("Datasets/olympia2016_athletes.csv")
df.head()

,id,name,nationality,sex,dob,height,weight,sport,gold,silver,bronze
0,736041664,A Jesus Garcia,ESP,male,10/17/69,1.72,64.0,athletics,0,0,0
1,532037425,A Lam Shin,KOR,female,9/23/86,1.68,56.0,fencing,0,0,0
2,435962603,Aaron Brown,CAN,male,5/27/92,1.98,79.0,athletics,0,0,1
3,521041435,Aaron Cook,MDA,male,1/2/91,1.83,80.0,taekwondo,0,0,0
4,33922579,Aaron Gate,NZL,male,11/26/90,1.81,71.0,cycling,0,0,0


In [3]:
df['total_medals'] = df['gold'] + df['silver'] + df['bronze']

country_stats = df.groupby("nationality").agg(
    num_athletes=("id", "count"),          
    total_gold=("gold", "sum"),
    total_silver=("silver", "sum"),
    total_bronze=("bronze", "sum"),
    total_medals=("total_medals", "sum")
).reset_index()

country_stats.head()

,nationality,num_athletes,total_gold,total_silver,total_bronze,total_medals
0,AFG,3,0,0,0,0
1,ALB,6,0,0,0,0
2,ALG,68,0,2,0,2
3,AND,5,0,0,0,0
4,ANG,26,0,0,0,0


In [4]:
max_all_medals = country_stats["total_medals"].max()
max_all_athletes = country_stats["num_athletes"].max()

print(f"Maximum number of medals across all countries: {max_all_medals}")
print(f"Maximum number of athletes across all countries: {max_all_athletes}")

Maximum number of medals across all countries: 264
Maximum number of athletes across all countries: 567


In [5]:
def get_plot(max_athletes, max_medals):
    filtered_countries = country_stats[
        (country_stats["num_athletes"] <= max_athletes) &
        (country_stats["total_medals"] <= max_medals)
    ]

    source = ColumnDataSource(data=dict(
        nationality = filtered_countries["nationality"],
        athlete_count = filtered_countries["num_athletes"],
        medal_total = filtered_countries["total_medals"],
        gold = filtered_countries["total_gold"],
        silver = filtered_countries["total_silver"],
        bronze = filtered_countries["total_bronze"]
    ))

    hover = HoverTool(tooltips=[
        ("Country", "@nationality"),
        ("Number of Athletes", "@athlete_count"),
        ("Total Medals", "@medal_total"),
        ("Gold", "@gold"),
        ("Silver", "@silver"),
        ("Bronze", "@bronze")
    ])

    plot = figure(
        title="Rio Olympics 2016 - Medal comparison",
        x_axis_label="Number of Medals",
        y_axis_label="Num of Athletes",
        tools=[hover, "pan", "box_zoom", "reset"]
    )

    plot.scatter(x="medal_total", y="athlete_count", source=source, size=8, color="teal", alpha=0.7)

    return plot

In [6]:
@interact(
    max_athletes = IntSlider(min=0, max=max_all_athletes, value=max_all_athletes, orientation="vertical"),
    max_medals = IntSlider(min=0, max=max_all_medals, value=max_all_medals, orientation="horizontal")
)
def display_visualization(max_athletes, max_medals):
    fig_output = get_plot(max_athletes, max_medals)
    handle = show(fig_output, notebook_handle=True)
    push_notebook(handle=handle)

interactive(children=(IntSlider(value=567, description='max_athletes', max=567, orientation='vertical'), IntSl…

In [7]:
static_fig = get_plot(max_all_athletes, max_all_medals)
show(static_fig)